In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df = spark.table("retailnew.bronze.store_inventory_api")

# Cast types
df = (
    df.withColumn("qty", F.col("qty").cast("int"))
      .withColumn("api_updated_at", F.to_timestamp("api_updated_at"))
      .withColumn("ingest_ts", F.to_timestamp("ingest_ts"))
)

# Data quality
df = df.filter(F.col("qty") >= 0)
df = df.filter(F.col("api_updated_at") <= F.current_timestamp())
df = df.filter(F.col("store_id").isNotNull() & F.col("product_id").isNotNull())

# Deduplicate: keep latest per store-product
w = Window.partitionBy("store_id", "product_id").orderBy(F.col("api_updated_at").desc())

df = (
    df.withColumn("rn", F.row_number().over(w))
      .filter(F.col("rn") == 1)
      .drop("rn")
)

# Add business logic
df = df.withColumn(
    "stock_status",
    F.when(F.col("qty") == 0, "OUT_OF_STOCK")
     .when(F.col("qty") < 5, "LOW_STOCK")
     .otherwise("IN_STOCK")
)

df = df.withColumn(
    "days_since_update",
    F.datediff(F.current_timestamp(), F.col("api_updated_at"))
)

# Write to Silver
(
    df.write
      .format("delta")
      .mode("overwrite")
      .option("overwriteSchema", "true")
      .saveAsTable("retailnew.silver.inventory_clean")
)
